In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor, VotingRegressor
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.impute import SimpleImputer
import matplotlib.pyplot as plt

In [3]:
df = pd.read_csv("../best_data.csv")

In [4]:
months = ['Month_1', 'Month_2', 'Month_3', 'Month_4', 'Month_5', 'Month_6','Month_7', 'Month_8', 'Month_9', 'Month_10', 'Month_11', 'Month_12']
features = ['Latitude_Degrees', 'Longitude_Degrees', 'Depth_m', 'ClimSST', 'SSTA', 'Percent_Bleaching']

In [5]:
df = df[features + months]

In [8]:
df.columns

Index(['Latitude_Degrees', 'Longitude_Degrees', 'Depth_m', 'ClimSST', 'SSTA',
       'Percent_Bleaching', 'Month_1', 'Month_2', 'Month_3', 'Month_4',
       'Month_5', 'Month_6', 'Month_7', 'Month_8', 'Month_9', 'Month_10',
       'Month_11', 'Month_12'],
      dtype='object')

In [9]:
df['Status'] = np.where(df['Percent_Bleaching'] <= 1, True, False)
X = df.drop(columns=["Percent_Bleaching", "Status"])   # ⭐ ลบทั้งคู่
Y = df["Status"]

In [13]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)
print("Class distribution:")
print(Y.value_counts())
print(Y.value_counts(normalize=True).round(3))
print(y_test[y_test == True].count()/y_test.count()*100)
print(y_train[y_train == True].count()/y_train.count()*100)

Class distribution:
Status
True     12088
False     9840
Name: count, dtype: int64
Status
True     0.551
False    0.449
Name: proportion, dtype: float64
55.608755129958965
55.005130543837645


In [26]:
import optuna
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

# Optuna

In [27]:
month_cols     = ['Month_1','Month_2','Month_3','Month_4','Month_5','Month_6',
                  'Month_7','Month_8','Month_9','Month_10','Month_11','Month_12']
paper_features = ['Latitude_Degrees', 'Longitude_Degrees', 'Depth_m', 'ClimSST', 'SSTA']

X_train_num    = X[paper_features].loc[X_train_raw.index].copy()  # 5 numeric cols
X_extras_train = X[month_cols].loc[X_train_raw.index].copy()      # 12 month OHE cols

# ============================================================
# Helper: CV with proper preprocessing within fold
# ============================================================
def cv_classif(model_factory, X_num, X_extras, y, cv_splits=5):
    """
    Stratified K-Fold CV — fits preprocessing inside each fold to prevent data leakage.

    Final feature vector = [5 scaled numeric] + [12 month OHE] = 17 features

    Args:
        model_factory : callable → returns a fresh model each fold
        X_num         : DataFrame of 5 numeric features (impute → scale)
        X_extras      : DataFrame of 12 month OHE columns (attached as-is)
        y             : target Series
        cv_splits     : number of StratifiedKFold splits
    Returns:
        Mean F1-weighted score across folds
    """
    X_num    = X_num.reset_index(drop=True)
    X_extras = X_extras.reset_index(drop=True)
    y        = y.reset_index(drop=True)

    skf    = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=42)
    scores = []

    for train_idx, val_idx in skf.split(X_num, y):
        # ----- split -----
        X_num_tr,  X_num_val = X_num.iloc[train_idx],    X_num.iloc[val_idx]
        X_ex_tr,   X_ex_val  = (X_extras.iloc[train_idx].reset_index(drop=True),
                                 X_extras.iloc[val_idx].reset_index(drop=True))
        y_tr,      y_val     = (y.iloc[train_idx].reset_index(drop=True),
                                 y.iloc[val_idx].reset_index(drop=True))

        # ----- fit preprocessing on train fold only ⭐ -----
        imputer  = SimpleImputer(strategy='median')
        X_tr_imp = imputer.fit_transform(X_num_tr)
        X_vl_imp = imputer.transform(X_num_val)

        scaler  = StandardScaler()
        X_tr_ss = scaler.fit_transform(X_tr_imp)
        X_vl_ss = scaler.transform(X_vl_imp)

        # ----- concat scaled numeric + month OHE -----
        # Final shape: [Lat, Lon, Depth, ClimSST, SSTA] + [Month_1 … Month_12] = 17 cols
        X_tr_final = pd.concat(
            [pd.DataFrame(X_tr_ss, columns=paper_features), X_ex_tr], axis=1
        )
        X_vl_final = pd.concat(
            [pd.DataFrame(X_vl_ss, columns=paper_features), X_ex_val], axis=1
        )

        # ----- train + evaluate -----
        model = model_factory()
        model.fit(X_tr_final, y_tr)
        y_pred = model.predict(X_vl_final)
        scores.append(f1_score(y_val, y_pred, average='weighted'))

    return np.mean(scores)

# ============================================================
# 1. Random Forest objective
# ============================================================
def objective_rf(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 200, 600),
        'max_depth':         trial.suggest_int('max_depth', 10, 50),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
        'min_samples_leaf':  trial.suggest_int('min_samples_leaf', 1, 5),
        'max_features':      trial.suggest_categorical('max_features', ['sqrt', None]),
        'random_state':      42,
        'n_jobs':            -1,
    }
    return cv_classif(
        lambda: RandomForestClassifier(**params),
        X_train_num,
        X_extras_train,
        y_train
    )

# ============================================================
# Run study
# ============================================================
studies  = {}
n_trials = 50

print("=" * 60)
print("🌲 Tuning Random Forest...")
print("=" * 60)
studies['rf'] = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42)
)
studies['rf'].optimize(objective_rf, n_trials=n_trials, show_progress_bar=True)

# ============================================================
# Summary
# ============================================================
print("\n" + "=" * 60)
print("📊 Best Results")
print("=" * 60)
for name, study in studies.items():
    print(f"\n{name.upper()}:")
    print(f"  Best F1:     {study.best_value:.4f}")
    print(f"  Best params: {study.best_params}")

[I 2026-05-17 11:01:15,344] A new study created in memory with name: no-name-2a749787-8454-40d8-bade-369b1426117b


🌲 Tuning Random Forest...


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-05-17 11:01:19,356] Trial 0 finished with value: 0.8832832715579635 and parameters: {'n_estimators': 350, 'max_depth': 48, 'min_samples_split': 8, 'min_samples_leaf': 3, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.8832832715579635.
[I 2026-05-17 11:01:25,074] Trial 1 finished with value: 0.8839838383570024 and parameters: {'n_estimators': 223, 'max_depth': 45, 'min_samples_split': 7, 'min_samples_leaf': 4, 'max_features': None}. Best is trial 1 with value: 0.8839838383570024.
[I 2026-05-17 11:01:38,921] Trial 2 finished with value: 0.909284168124519 and parameters: {'n_estimators': 533, 'max_depth': 18, 'min_samples_split': 3, 'min_samples_leaf': 1, 'max_features': None}. Best is trial 2 with value: 0.909284168124519.
[I 2026-05-17 11:01:49,150] Trial 3 finished with value: 0.9006020252140535 and parameters: {'n_estimators': 373, 'max_depth': 21, 'min_samples_split': 7, 'min_samples_leaf': 1, 'max_features': None}. Best is trial 2 with value: 0.909284168124519.
[I 20